# Stage 1 Data Cleaning — Lecture-Level Attendance Dataset
**SYMCA Div A+B | SEM III | MCA | A.Y. 2026-27**

Steps performed:
1. Report missing values per column
2. Compute `Attendance_Percentage`
3. Flag impossible values (Present > Enrolled, negatives)
4. Convert `Date` to datetime, `Day_of_Week` to Categorical
5. Save cleaned output as `attendance_stage1.csv`

In [1]:
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')

DATA_PATH = r'D:\Data_Science_attendence_project\classroom-attendance-schedule-project\data\processed\lecture_level_dataset.csv'
OUT_PATH  = r'D:\Data_Science_attendence_project\classroom-attendance-schedule-project\data\processed\attendance_stage1.csv'

df_raw = pd.read_csv(DATA_PATH)
print(f'Loaded {len(df_raw)} rows x {len(df_raw.columns)} columns')
df_raw.head(3)

Loaded 19 rows x 18 columns


,Date,Day_of_Week,Lecture_Number,Start_Time,End_Time,Approx_Start_Time,Approx_End_Time,Subject,Faculty_ID,Semester,Branch,Section,Classroom,Total_Enrolled_Students,Students_Present_Count,Confidence,Notes,Match_Type
0,25-06-2026,Thursday,1,08:30,09:15,08:15,09:15,Mobile Application Development,F_01+F_13,III,MCA,A+B,Computer Lab,NaN,46,medium,NaN,exact
1,29-06-2026,Monday,1,08:30,09:15,08:15,08:45,Mobile Application Development,F_01+F_13,III,MCA,A+B,Computer Lab,NaN,31,medium,NaN,exact
2,30-06-2026,Tuesday,1,08:30,09:15,08:15,09:30,Mobile Application Development,F_01+F_13,III,MCA,A+B,Computer Lab,NaN,19,medium,NaN,exact


## 1. Missing Values — Before Cleaning

In [2]:
null_before = df_raw.isnull().sum().rename('Null_Count')
null_pct    = (df_raw.isnull().mean()*100).round(2).rename('Null_Pct')
null_report = pd.concat([null_before, null_pct], axis=1)
null_report['Dtype'] = df_raw.dtypes
print('=== NULL COUNTS BEFORE CLEANING ===')
print(null_report.to_string())

=== NULL COUNTS BEFORE CLEANING ===
                         Null_Count  Null_Pct    Dtype
Date                              0      0.00   object
Day_of_Week                       0      0.00   object
Lecture_Number                    0      0.00    int64
Start_Time                        0      0.00   object
End_Time                          0      0.00   object
Approx_Start_Time                 0      0.00   object
Approx_End_Time                   1      5.26   object
Subject                           0      0.00   object
Faculty_ID                        0      0.00   object
Semester                          0      0.00   object
Branch                            0      0.00   object
Section                           0      0.00   object
Classroom                         0      0.00   object
Total_Enrolled_Students          19    100.00  float64
Students_Present_Count            0      0.00    int64
Confidence                        0      0.00   object
Notes                        

## 2. Type Conversions
- `Date` -> `datetime64`
- `Day_of_Week` -> ordered Categorical (Mon-Sat)

In [3]:
df = df_raw.copy()

# --- Date -> datetime ---
df['Date'] = pd.to_datetime(df['Date'], format='%d-%m-%Y', errors='coerce')

# --- Day_of_Week -> ordered Categorical ---
day_order = ['Monday','Tuesday','Wednesday','Thursday','Friday','Saturday']
df['Day_of_Week'] = pd.Categorical(df['Day_of_Week'], categories=day_order, ordered=True)

# --- OFFICIAL class strength (confirmed by faculty/admin) ---
# Context: 200 = combined Div A+B strength.
# This register covers ONE division only => per-division strength = 80.
OFFICIAL_STRENGTH = 80
df['Total_Enrolled_Students'] = OFFICIAL_STRENGTH

# --- Flag stray/incomplete entry (11-Aug-2026, count=1) ---
stray_mask = (df['Date'] == '2026-08-11') & (df['Students_Present_Count'] == 1)
df['Flag_Stray_Entry'] = stray_mask

print(f'OFFICIAL_STRENGTH set to: {OFFICIAL_STRENGTH} (per division)')
print(f'Stray entries flagged  : {stray_mask.sum()}')
print(df[['Date','Day_of_Week','Total_Enrolled_Students','Flag_Stray_Entry']].dtypes)

OFFICIAL_STRENGTH set to: 80 (per division)
Stray entries flagged  : 1
Date                       datetime64[ns]
Day_of_Week                      category
Total_Enrolled_Students             int64
Flag_Stray_Entry                     bool
dtype: object


## 3. Compute `Attendance_Percentage`

In [4]:
df['Attendance_Percentage'] = (
    df['Students_Present_Count'] / df['Total_Enrolled_Students'] * 100
).round(2)

print('Attendance_Percentage -- descriptive stats:')
print(df['Attendance_Percentage'].describe().round(2))

Attendance_Percentage -- descriptive stats:
count    19.00
mean     36.78
std      23.04
min       1.25
25%      18.12
50%      35.00
75%      55.62
max      75.00
Name: Attendance_Percentage, dtype: float64


## 4. Flag Impossible Values
Rules:
- `Students_Present_Count > Total_Enrolled_Students`
- `Students_Present_Count < 0`

In [5]:
mask_over     = df['Students_Present_Count'] > df['Total_Enrolled_Students']
mask_negative = df['Students_Present_Count'] < 0
mask_impossible = mask_over | mask_negative
df['Flag_Impossible'] = mask_impossible

n_bad = int(mask_impossible.sum())
print(f'Rows flagged IMPOSSIBLE: {n_bad}')

if n_bad > 0:
    flag_cols = ['Date','Day_of_Week','Subject','Students_Present_Count',
                 'Total_Enrolled_Students','Attendance_Percentage','Confidence','Notes']
    print(df[mask_impossible][flag_cols].to_string(index=False))
else:
    print('No impossible values found with provisional Enrolled = 60.')
    at_100 = df[df['Students_Present_Count'] == df['Total_Enrolled_Students']]
    print(f'Rows at 100 pct attendance (verify these): {len(at_100)}')
    print(at_100[['Date','Subject','Students_Present_Count','Confidence']].to_string(index=False))

Rows flagged IMPOSSIBLE: 0
No impossible values found with provisional Enrolled = 60.
Rows at 100 pct attendance (verify these): 0
Empty DataFrame
Columns: [Date, Subject, Students_Present_Count, Confidence]
Index: []


## 5. Low-Confidence and Provisional Rows

In [6]:
print('=== ROWS BY CONFIDENCE LEVEL ===')
print(df['Confidence'].value_counts().to_string())

print('\n=== LOW CONFIDENCE ROWS (verify before final use) ===')
low_cols = ['Date','Day_of_Week','Subject','Students_Present_Count','Notes','Match_Type']
print(df[df['Confidence']=='low'][low_cols].to_string(index=False))

print('\n=== UNMATCHED TIMETABLE SLOTS ===')
unmatched = df[df['Match_Type']=='unmatched']
print(f'{len(unmatched)} unmatched rows')
if len(unmatched):
    print(unmatched[['Date','Approx_Start_Time','Confidence','Notes']].to_string(index=False))

=== ROWS BY CONFIDENCE LEVEL ===
Confidence
medium    13
low        4
high       2

=== LOW CONFIDENCE ROWS (verify before final use) ===
      Date Day_of_Week                        Subject  Students_Present_Count                                       Notes Match_Type
2026-07-22   Wednesday Mobile Application Development                      15                           verify date digit      exact
2026-07-23    Thursday Mobile Application Development                      30                           verify date digit     approx
2026-07-30    Thursday Mobile Application Development                      60                           verify page split      exact
2026-08-06    Thursday Mobile Application Development                      43 has two sessions - split not fully verified     approx

=== UNMATCHED TIMETABLE SLOTS ===
0 unmatched rows


## 6. First 15 Rows of Cleaned Dataset

In [7]:
display_cols = ['Date','Day_of_Week','Lecture_Number','Start_Time','Subject',
                'Faculty_ID','Students_Present_Count','Total_Enrolled_Students',
                'Attendance_Percentage','Confidence','Match_Type','Flag_Impossible']
print(df[display_cols].head(15).to_string(index=True))

         Date Day_of_Week  Lecture_Number Start_Time                         Subject           Faculty_ID  Students_Present_Count  Total_Enrolled_Students  Attendance_Percentage Confidence Match_Type  Flag_Impossible
0  2026-06-25    Thursday               1      08:30  Mobile Application Development            F_01+F_13                      46                       80                  57.50     medium      exact            False
1  2026-06-29      Monday               1      08:30  Mobile Application Development            F_01+F_13                      31                       80                  38.75     medium      exact            False
2  2026-06-30     Tuesday               1      08:30  Mobile Application Development            F_01+F_13                      19                       80                  23.75     medium      exact            False
3  2026-07-02    Thursday               1      08:30  Mobile Application Development            F_01+F_13                      27   

## 7. Null Counts — Before vs After Cleaning

In [8]:
null_after = df.isnull().sum().rename('Null_After')
comparison = pd.concat([null_before, null_after], axis=1)
comparison.columns = ['Null_Before','Null_After']
comparison['Delta'] = comparison['Null_Before'] - comparison['Null_After']
print('=== NULL COUNT COMPARISON ===')
changed = comparison[comparison['Null_Before'] > 0]
print(changed.to_string())

=== NULL COUNT COMPARISON ===
                         Null_Before  Null_After  Delta
Approx_End_Time                  1.0           1    0.0
Total_Enrolled_Students         19.0           0   19.0
Notes                           14.0          14    0.0


## 8. Save `attendance_stage1.csv`

In [9]:
df.to_csv(OUT_PATH, index=False)
print(f'Saved: {OUT_PATH}')
print(f'Final shape: {df.shape}')
print('\nColumn dtypes:')
print(df.dtypes.to_string())

Saved: D:\Data_Science_attendence_project\classroom-attendance-schedule-project\data\processed\attendance_stage1.csv
Final shape: (19, 21)

Column dtypes:
Date                       datetime64[ns]
Day_of_Week                      category
Lecture_Number                      int64
Start_Time                         object
End_Time                           object
Approx_Start_Time                  object
Approx_End_Time                    object
Subject                            object
Faculty_ID                         object
Semester                           object
Branch                             object
Section                            object
Classroom                          object
Total_Enrolled_Students             int64
Students_Present_Count              int64
Confidence                         object
Notes                              object
Match_Type                         object
Flag_Stray_Entry                     bool
Attendance_Percentage             float64
Flag_